In [2]:
# =============================================================================
# INDIVIDUAL METHOD RUNNER
# Quick testing of a single method on a single dataset
# =============================================================================

# -----------------------------------------------------------------------------
# CONFIGURATION - CHANGE THESE VALUES
# -----------------------------------------------------------------------------

METHOD = "xgboost"           # Method to run (e.g., 'xgboost', 'catboost', 'tabpfn', 'mlp')
DATASET = "0006.lgd_freddie"        # Dataset name (e.g., '0014.hmeq', '0001.gmsc')
TASK = "lgd"                  # Task type: 'pd' (classification) or 'lgd' (regression)

# -----------------------------------------------------------------------------
# FIXED SETTINGS (for quick testing)
# -----------------------------------------------------------------------------

ROW_LIMIT = 10000             # Limit rows for fast execution
MAX_EPOCHS = 15              # Max epochs for deep learning methods
CV_SPLITS = 3                # Number of folds for cross-validation
TUNE = True                 # No HPO or HPO? 
SEED = 42                    # Random seed
TEST_SIZE = 0.2              # Test set fraction
VAL_SIZE = 0.2               # Validation set fraction

# -----------------------------------------------------------------------------
# SETUP
# -----------------------------------------------------------------------------

import sys
from pathlib import Path
import pickle
import json
from datetime import datetime

# Add project root to path (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"\n{'='*60}")
print(f" Running: {METHOD} on {DATASET} ({TASK.upper()})")
print(f"{'='*60}")
print(f"  Row limit:  {ROW_LIMIT}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  CV splits:  {CV_SPLITS}")
print(f"  HPO:        {TUNE}")
print(f"{'='*60}\n")

# -----------------------------------------------------------------------------
# RUN METHOD
# -----------------------------------------------------------------------------

from src.methods.method_runner import run_talent_method, get_available_methods

# Show available methods
available = get_available_methods()
print(f"Available classical methods: {available['classical']}")
print(f"Available deep methods: {available['deep'][:10]}... ({len(available['deep'])} total)")
print()

# Run the method
results = run_talent_method(
    task=TASK,
    dataset=DATASET,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    cv_splits=CV_SPLITS,
    seed=SEED,
    row_limit=ROW_LIMIT,
    method=METHOD,
    max_epoch=MAX_EPOCHS,
    tune=TUNE,
    verbose=True,
)

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------

print(f"\n{'='*60}")
print(f" RESULTS")
print(f"{'='*60}")

for fold_id, fold_results in results.items():
    print(f"\nFold {fold_id}:")
    print(f"  Train time: {fold_results['train_time']:.2f}s")
    print(f"  Samples:    {len(fold_results['y_true'])}")
    
    if TASK == 'lgd':
        print(f"  Clipped:    {fold_results['n_clipped_below']} below, {fold_results['n_clipped_above']} above")
    
    print(f"\n  Metrics:")
    for metric_name, metric_value in fold_results['metrics'].items():
        if not (isinstance(metric_value, float) and metric_value != metric_value):  # Skip NaN
            print(f"    {metric_name:20s}: {metric_value:.4f}")

# -----------------------------------------------------------------------------
# SAVE RESULTS
# -----------------------------------------------------------------------------

# Create output directory
output_dir = PROJECT_ROOT / 'results' / 'individual_method_runner'
output_dir.mkdir(parents=True, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"{METHOD}_{DATASET}_{TASK}_{timestamp}"

# Save as pickle (full results)
pickle_path = output_dir / f"{filename}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to: {pickle_path}")

# Save summary as JSON (metrics only, for easy viewing)
summary = {
    'method': METHOD,
    'dataset': DATASET,
    'task': TASK,
    'timestamp': timestamp,
    'config': {
        'row_limit': ROW_LIMIT,
        'max_epochs': MAX_EPOCHS,
        'cv_splits': CV_SPLITS,
        'tune': TUNE,
        'seed': SEED,
    },
    'folds': {}
}

for fold_id, fold_results in results.items():
    summary['folds'][fold_id] = {
        'train_time': fold_results['train_time'],
        'n_samples': len(fold_results['y_true']),
        'metrics': {k: v for k, v in fold_results['metrics'].items() if not (isinstance(v, float) and v != v)},
    }
    if TASK == 'lgd':
        summary['folds'][fold_id]['n_clipped_below'] = fold_results['n_clipped_below']
        summary['folds'][fold_id]['n_clipped_above'] = fold_results['n_clipped_above']

json_path = output_dir / f"{filename}.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {json_path}")

print(f"\n{'='*60}")
print(f" DONE")
print(f"{'='*60}")

[I 2025-12-24 09:43:58,525] A new study created in memory with name: no-name-cde31128-0228-4bd7-9461-b2b0d8d54abe


Project root: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit

 Running: xgboost on 0006.lgd_freddie (LGD)
  Row limit:  10000
  Max epochs: 15
  CV splits:  3
  HPO:        True

Available classical methods: ['LinearRegression', 'LogReg', 'NCM', 'NaiveBayes', 'RandomForest', 'catboost', 'dummy', 'knn', 'lightgbm', 'svm', 'xgboost']
Available deep methods: ['amformer', 'autoint', 'bishop', 'danets', 'dcn2', 'dnnr', 'excelformer', 'ftt', 'grande', 'grownet']... (38 total)


Running xgboost (classical) on 0006.lgd_freddie (LGD)

[HPO] Per-fold hyperparameter optimization enabled
[HPO] Each fold: 50 trials (optimizes on validation loss)

Preparing data with 3 CV splits...
Fold IDs: [1, 2, 3]

Directory setup:
  Config directory: C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\config_hpo\lgd\0006.lgd_freddie\xgboost\HPO_PER_FOLD
  Merged config: xgboost-all-folds.json

Fold 1/3
using gpu: 0
{'cat_min_frequency': 0.0,

  0%|          | 0/50 [00:00<?, ?it/s]

[0]	validation_0-rmse:0.97934
[1]	validation_0-rmse:0.95784
[2]	validation_0-rmse:0.93932
[3]	validation_0-rmse:0.92398
[4]	validation_0-rmse:0.91379
[5]	validation_0-rmse:0.90193
[6]	validation_0-rmse:0.88989
[7]	validation_0-rmse:0.88165
[8]	validation_0-rmse:0.87489
[9]	validation_0-rmse:0.86998
[10]	validation_0-rmse:0.86498
[11]	validation_0-rmse:0.86105
[12]	validation_0-rmse:0.85623
[13]	validation_0-rmse:0.85095
[14]	validation_0-rmse:0.84734
[15]	validation_0-rmse:0.84467
[16]	validation_0-rmse:0.84178
[17]	validation_0-rmse:0.83809
[18]	validation_0-rmse:0.83527
[19]	validation_0-rmse:0.83254
[20]	validation_0-rmse:0.83074
[21]	validation_0-rmse:0.82971
[22]	validation_0-rmse:0.82807
[23]	validation_0-rmse:0.82668
[24]	validation_0-rmse:0.82537
[25]	validation_0-rmse:0.82377
[26]	validation_0-rmse:0.82062
[27]	validation_0-rmse:0.81868
[28]	validation_0-rmse:0.81725
[29]	validation_0-rmse:0.81687
[30]	validation_0-rmse:0.81601
[31]	validation_0-rmse:0.81530
[32]	validation_0-

Best trial: 0. Best value: 0.255742:   2%|▏         | 1/50 [00:00<00:27,  1.79it/s]

...

Best trial: 0. Best value: 0.256109:  98%|█████████▊| 49/50 [00:32<00:00,  1.26it/s]

[I 2025-12-24 09:45:38,667] Trial 48 finished with value: 0.29089621137033816 and parameters: {'optional_alpha': False, 'colsample_bylevel': 0.7535038351366237, 'colsample_bytree': 0.6428861259884232, 'optional_gamma': False, 'optional_lambda': True, 'lambda': 6.551293621711666, 'learning_rate': 0.006450612358007096, 'max_depth': 4, 'min_child_weight': 0.02185509809669761, 'subsample': 0.863807368355408, 'n_bins': 242}. Best is trial 0 with value: 0.25610928523559134.
[0]	validation_0-rmse:0.94088
[1]	validation_0-rmse:0.90007
[2]	validation_0-rmse:0.87102
[3]	validation_0-rmse:0.86185
[4]	validation_0-rmse:0.84785
[5]	validation_0-rmse:0.84112
[6]	validation_0-rmse:0.83282
[7]	validation_0-rmse:0.83189
[8]	validation_0-rmse:0.82858
[9]	validation_0-rmse:0.82196
[10]	validation_0-rmse:0.82009
[11]	validation_0-rmse:0.81517
[12]	validation_0-rmse:0.81343
[13]	validation_0-rmse:0.81226
[14]	validation_0-rmse:0.81235
[15]	validation_0-rmse:0.81159
[16]	validation_0-rmse:0.81161
[17]	valid

Best trial: 0. Best value: 0.256109: 100%|██████████| 50/50 [00:33<00:00,  1.51it/s]

[I 2025-12-24 09:45:39,393] Trial 49 finished with value: 0.2646675102247929 and parameters: {'optional_alpha': True, 'alpha': 0.23029925900054607, 'colsample_bylevel': 0.8680331719188861, 'colsample_bytree': 0.5978200665063821, 'optional_gamma': False, 'optional_lambda': False, 'learning_rate': 0.6418739233385424, 'max_depth': 7, 'min_child_weight': 339.28840727299547, 'subsample': 0.7401107509580026, 'n_bins': 165}. Best is trial 0 with value: 0.25610928523559134.
Best Hyper-Parameters
{'model': {'alpha': 0.010656970429469137, 'colsample_bylevel': 0.7724415914984484, 'colsample_bytree': 0.7118273996694524, 'gamma': 0, 'lambda': 6.829913261377665e-05, 'learning_rate': 0.09091283280651452, 'max_depth': 7, 'min_child_weight': 0.2424260549741265, 'subsample': 0.9627983191463305}, 'fit': {'n_bins': 20}}
[HPO] Config saved (fold 3)
[HPO] Best hyperparameters: {'alpha': 0.010656970429469137, 'colsample_bylevel': 0.7724415914984484, 'colsample_bytree': 0.7118273996694524, 'gamma': 0, 'lambda

[0]	validation_0-rmse:0.97660
[1]	validation_0-rmse:0.95664
[2]	validation_0-rmse:0.93955
[3]	validation_0-rmse:0.92738
[4]	validation_0-rmse:0.91532
[5]	validation_0-rmse:0.90276
[6]	validation_0-rmse:0.89092
[7]	validation_0-rmse:0.88228
[8]	validation_0-rmse:0.87438
[9]	validation_0-rmse:0.86929
[10]	validation_0-rmse:0.86483
[11]	validation_0-rmse:0.86043
[12]	validation_0-rmse:0.85416
[13]	validation_0-rmse:0.85042
[14]	validation_0-rmse:0.84704
[15]	validation_0-rmse:0.84446
[16]	validation_0-rmse:0.84110
[17]	validation_0-rmse:0.83875
[18]	validation_0-rmse:0.83671
[19]	validation_0-rmse:0.83424
[20]	validation_0-rmse:0.83283
[21]	validation_0-rmse:0.83070
[22]	validation_0-rmse:0.82890
[23]	validation_0-rmse:0.82688
[24]	validation_0-rmse:0.82475
[25]	validation_0-rmse:0.82198
[26]	validation_0-rmse:0.82064
[27]	validation_0-rmse:0.81931
[28]	validation_0-rmse:0.81798
[29]	validation_0-rmse:0.81745
[30]	validation_0-rmse:0.81717
[31]	validation_0-rmse:0.81728
[32]	validation_0-